# Oracle Spark Connection

Objetivo:
- Iniciar a sessao do Spark.
- Conectar no Oracle via JDBC usando o IP (evita problema de DNS no JVM).
- Fazer uma leitura simples para validar.

Requisitos:
- Driver Oracle JDBC disponivel no classpath do Spark (ojdbc).


In [10]:
from pyspark.sql import SparkSession

# Configurações otimizadas para Spark
spark = (
    SparkSession.builder
    .appName("oracle-connection-check")
    .config("spark.sql.shuffle.partitions", "8")  # Reduzir para datasets pequenos
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.adaptive.enabled", "true")  # Adaptive Query Execution
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryoserializer.buffer.max", "512m")
    .getOrCreate()
)

print("✓ Spark Session criada com otimizações")

✓ Spark Session criada com otimizações


## Configuracao Oracle

Ajuste usuario e senha.


In [11]:
oracle_host = "10.255.150.11"
oracle_port = 1521
oracle_service = "bi.grupotracker.com.br"

oracle_user = "clickhouse"
oracle_password = "qiU!EOoe"

jdbc_url = f"jdbc:oracle:thin:@//{oracle_host}:{oracle_port}/{oracle_service}"
jdbc_url


'jdbc:oracle:thin:@//10.255.150.11:1521/bi.grupotracker.com.br'

## Teste rapido de conectividade TCP (opcional)

Se falhar aqui, nao adianta tentar JDBC.


In [12]:
import socket

def check_tcp(host: str, port: int, timeout: float = 5.0) -> bool:
    sock = socket.socket()
    sock.settimeout(timeout)
    try:
        sock.connect((host, int(port)))
        print(f"OK: {host}:{port}")
        return True
    except Exception as e:
        print(f"FAIL: {host}:{port} -> {e}")
        return False
    finally:
        sock.close()

check_tcp(oracle_host, oracle_port)


OK: 10.255.150.11:1521


True

## Leitura simples no Oracle

Usa `query` para validar que o JDBC funciona.


In [13]:
df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("user", oracle_user)
    .option("password", oracle_password)
    .option("driver", "oracle.jdbc.OracleDriver")
    .option("query", "SELECT 1 AS ok FROM dual")
    .load()
)

df.show()


+------------+
|          OK|
+------------+
|1.0000000000|
+------------+



## Pipeline de extracao (11 tabelas)

O objetivo aqui e apenas validar leitura. Para cada tabela, fazemos um `show(5)`.


In [14]:
tables_to_extract = [
    "ginf.depara_cliente",
    "ginf.BASE_CEP_COMPLETA",
    "ginf.TST_CONTRATOS_BI",
    "siga.SC5030",
    "siga.SC6030",
    "ginf.TST_HISTORICO_SOLICITACOES",
    "ginf.TST_SOLICIT_CADASTRADAS",
    "siga.SD2030",
    "siga.SF2030",
    "siga.ZTX030",
    "ginf.TST_CONTRATOS",
]

print(f"Total de tabelas para extrair: {len(tables_to_extract)}")
for t in tables_to_extract:
    print(f"  - {t}")


Total de tabelas para extrair: 11
  - ginf.depara_cliente
  - ginf.BASE_CEP_COMPLETA
  - ginf.TST_CONTRATOS_BI
  - siga.SC5030
  - siga.SC6030
  - ginf.TST_HISTORICO_SOLICITACOES
  - ginf.TST_SOLICIT_CADASTRADAS
  - siga.SD2030
  - siga.SF2030
  - siga.ZTX030
  - ginf.TST_CONTRATOS


In [15]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

def get_table_info(table):
    """Função para obter informações de uma tabela"""
    try:
        start_time = datetime.now()
        
        # Usar query COUNT otimizada ao invés de df.count()
        count_query = f"(SELECT COUNT(*) as cnt FROM {table}) tmp"
        
        count_df = (
            spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("user", oracle_user)
            .option("password", oracle_password)
            .option("driver", "oracle.jdbc.OracleDriver")
            .option("dbtable", count_query)
            .load()
        )
        
        row_count = count_df.first()[0]
        
        # Obter schema sem carregar dados
        schema_df = (
            spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("user", oracle_user)
            .option("password", oracle_password)
            .option("driver", "oracle.jdbc.OracleDriver")
            .option("dbtable", f"(SELECT * FROM {table} WHERE 1=0) tmp")
            .load()
        )
        
        col_count = len(schema_df.columns)
        
        duration = (datetime.now() - start_time).total_seconds()
        
        return {
            'Tabela': table,
            'Linhas': f'{row_count:,}',
            'Colunas': col_count,
            'Tempo (s)': f'{duration:.2f}',
            'Status': '✅'
        }
        
    except Exception as e:
        return {
            'Tabela': table,
            'Linhas': '-',
            'Colunas': '-',
            'Tempo (s)': '-',
            'Status': f'❌ {str(e)[:50]}'
        }

print("🔍 COLETANDO INFORMAÇÕES DAS TABELAS (PARALELO)...\n")

start_total = datetime.now()
table_info = []

# Processar 4 tabelas em paralelo
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(get_table_info, table): table for table in tables_to_extract}
    
    for future in as_completed(futures):
        table = futures[future]
        result = future.result()
        table_info.append(result)
        print(f"✓ {result['Tabela']}: {result['Linhas']} linhas em {result['Tempo (s)']}s")

duration_total = (datetime.now() - start_total).total_seconds()

# Ordenar por ordem original
table_info_sorted = sorted(table_info, key=lambda x: tables_to_extract.index(x['Tabela']))

# Exibir resumo
summary_df = pd.DataFrame(table_info_sorted)
display(summary_df)

success = summary_df[summary_df['Status'] == '✅'].shape[0]
errors = summary_df[summary_df['Status'] != '✅'].shape[0]

print(f"\n{'='*60}")
print(f"📊 RESUMO")
print(f"{'='*60}")
print(f"Total: {len(table_info)} tabelas")
print(f"✅ Sucesso: {success}")
print(f"❌ Erros: {errors}")
print(f"⏱️  Tempo total: {duration_total:.2f}s")
print(f"{'='*60}")

🔍 COLETANDO INFORMAÇÕES DAS TABELAS (PARALELO)...

✓ ginf.TST_CONTRATOS_BI: - linhas em -s
✓ ginf.BASE_CEP_COMPLETA: 96,778.0000000000 linhas em 1.03s
✓ ginf.depara_cliente: 47.0000000000 linhas em 1.05s
✓ siga.SC5030: 5,360,714.0000000000 linhas em 1.68s
✓ siga.SC6030: 7,559,160.0000000000 linhas em 1.88s
✓ siga.SD2030: 4,458,900.0000000000 linhas em 2.10s
✓ ginf.TST_SOLICIT_CADASTRADAS: 7,474,466.0000000000 linhas em 2.87s
✓ siga.SF2030: 2,089,997.0000000000 linhas em 1.69s
✓ siga.ZTX030: 8,757,508.0000000000 linhas em 2.73s
✓ ginf.TST_HISTORICO_SOLICITACOES: 50,184,474.0000000000 linhas em 5.73s
✓ ginf.TST_CONTRATOS: 3,046,546.0000000000 linhas em 8.47s


,Tabela,Linhas,Colunas,Tempo (s),Status
0,ginf.depara_cliente,47.0000000000,11,1.05,✅
1,ginf.BASE_CEP_COMPLETA,"96,778.0000000000",3,1.03,✅
2,ginf.TST_CONTRATOS_BI,-,-,-,❌ An error occurred while calling o181.load.\n...
3,siga.SC5030,"5,360,714.0000000000",188,1.68,✅
4,siga.SC6030,"7,559,160.0000000000",188,1.88,✅
5,ginf.TST_HISTORICO_SOLICITACOES,"50,184,474.0000000000",24,5.73,✅
6,ginf.TST_SOLICIT_CADASTRADAS,"7,474,466.0000000000",76,2.87,✅
7,siga.SD2030,"4,458,900.0000000000",311,2.10,✅
8,siga.SF2030,"2,089,997.0000000000",234,1.69,✅
9,siga.ZTX030,"8,757,508.0000000000",45,2.73,✅



📊 RESUMO
Total: 11 tabelas
✅ Sucesso: 10
❌ Erros: 1
⏱️  Tempo total: 12.39s
